# NIFTy8 sampling for an analytic scattering-statistic posterior

This notebook samples the distribution whose negative log-density is

$$
E(x) = \frac12\left[\Phi(x)-\mu\right]^T C^{-1}\left[\Phi(x)-\mu\right],
$$

where `Phi(x)` is computed with the STL scattering backend and the same target-derived normalization used when estimating `mu` and `C`.

The critical rule from `scattering_vi/README.md` is followed here: rebuild the STL target operator from the original target map, keep that operator, and evaluate all candidate maps with `norm="load_ref"` through that operator.

## Notes

- Run this notebook in the Python environment that has `nifty8`, `torch`, `numpy`, and the STL dependencies installed.
- The default paths use the local LSS covariance payload at `Nifty8/lss_cov/lss_covariance.npz` and its matching target map `Nifty8/lss_cov/lss.npy`.
- The custom NIFTy energy below uses exact PyTorch/STL gradients for the energy. For NIFTy sampling it also supplies a positive diagonal metric approximation; this keeps the NIFTy minimizers and MGVI-style sampler operational for a custom non-linear statistic likelihood.

In [1]:
from __future__ import annotations

import json
import sys
from argparse import Namespace
from pathlib import Path

import matplotlib.pyplot as plt
import nifty8 as ift
import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "STL-Dev").exists():
    # If the notebook is launched from STL-Dev/scattering_vi, move back to the repo root.
    PROJECT_ROOT = Path.cwd().parents[1]

SCATTERING_VI = PROJECT_ROOT / "STL-Dev" / "scattering_vi"
STL_REPO = PROJECT_ROOT / "STL-Dev"
for path in (str(SCATTERING_VI), str(STL_REPO)):
    if path not in sys.path:
        sys.path.insert(0, path)

from estimate_scattering_covariance import (  # noqa: E402
    DataClass,
    apply_standardized_st,
    configure_backend,
    load_target,
    target_statistics,
    torch_dtype,
)
from nifty8.operators.simple_linear_operators import VdotOperator  # noqa: E402

/Users/tsouros/Desktop/Projects/STL-Dev/STL_main/__init__.py:9: UserWarning: In development mode: Please run 'pip install -e .' in the root directory of the STL repository to properly install the package, its dependencies and enable proper version tracking.
  warnings.warn(


In [2]:
# Paths. These must refer to the same covariance-estimation run.
COVARIANCE_NPZ = PROJECT_ROOT / "Nifty8" / "lss_cov" / "lss_covariance.npz"
COVARIANCE_JSON = PROJECT_ROOT / "Nifty8" / "lss_cov" / "lss_covariance.json"
TARGET_INPUT = PROJECT_ROOT / "Nifty8" / "lss_cov" / "lss.npy"

# Runtime controls. Use CPU for portability; switch to cuda:0 on a GPU node.
DEVICE = "cpu"
DTYPE = "float64"
SEED = 7

# NIFTy controls.
USE_CORRELATED_FIELD_PRIOR = False  # False samples x directly with a standard-normal field prior.
RUN_MAP = True
RUN_VI = True
USE_GEOVI = False
N_POSTERIOR_SAMPLES = 8
VI_ITERATIONS = 4

# Numerical stabilization of the statistic covariance.
COVARIANCE_JITTER_REL = 1e-6
METRIC_STRENGTH = 1.0

In [3]:
def args_from_covariance_json(json_path: Path, input_path: Path, *, device: str, dtype: str) -> Namespace:
    """Reconstruct the scattering settings used by estimate_scattering_covariance.py."""
    if json_path.exists():
        with json_path.open("r", encoding="utf-8") as f:
            metadata = json.load(f)
        config = dict(metadata.get("config", {}))
    else:
        config = {}

    defaults = {
        "input": input_path,
        "target_size": 256,
        "subtract_mean": False,
        "device": device,
        "dtype": dtype,
        "J": 7,
        "L": 4,
        "wtype": "Bump-Steerable",
        "iso": True,
        "angular_ft": True,
        "harmonics_angle": 2,
        "scale_ft": True,
        "harmonics_scale": 3,
        "dj": 3,
        "compute_ps": False,
        "fewer_convolutions": False,
        "pbc": True,
        "stats_chunk_size": 32,
    }
    for key, value in defaults.items():
        config.setdefault(key, value)

    config["input"] = Path(input_path)
    config["device"] = device
    config["dtype"] = dtype
    return Namespace(**config)


def load_scattering_gaussian(npz_path: Path, jitter_rel: float = 1e-6):
    payload = np.load(npz_path)
    mu = payload["synthesized_mean"] if "synthesized_mean" in payload.files else payload["target_statistics"]
    if "covariance_within_batch" in payload.files:
        covariance = payload["covariance_within_batch"]
    elif "covariance_pooled" in payload.files:
        covariance = payload["covariance_pooled"]
    else:
        covariance = np.cov(payload["synthesized_statistics"], rowvar=False)

    active = payload["active_statistic_mask"] if "active_statistic_mask" in payload.files else np.ones_like(mu, dtype=bool)
    mu = np.asarray(mu, dtype=np.float64)[active]
    covariance = np.asarray(covariance, dtype=np.float64)[np.ix_(active, active)]
    covariance = 0.5 * (covariance + covariance.T)

    diag_scale = float(np.nanmedian(np.diag(covariance)))
    if not np.isfinite(diag_scale) or diag_scale <= 0:
        diag_scale = float(np.nanmax(np.diag(covariance)))
    if not np.isfinite(diag_scale) or diag_scale <= 0:
        diag_scale = 1.0
    covariance = covariance + (jitter_rel * diag_scale) * np.eye(covariance.shape[0])

    precision = np.linalg.pinv(covariance, hermitian=True)
    return mu, covariance, precision, active

In [4]:
args = args_from_covariance_json(COVARIANCE_JSON, TARGET_INPUT, device=DEVICE, dtype=DTYPE)
device = torch.device(args.device)
dtype = torch_dtype(args.dtype)
configure_backend(device, dtype)
ift.random.push_sseq_from_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)

mu_np, covariance_np, precision_np, active_mask_np = load_scattering_gaussian(
    COVARIANCE_NPZ,
    jitter_rel=COVARIANCE_JITTER_REL,
)

target_np, input_info = load_target(args)
target_t = torch.as_tensor(target_np.copy(), device=device, dtype=dtype)
target_stats, target_op, target_flat_t = target_statistics(target_t, args)

print(f"target: {TARGET_INPUT}")
print(f"target shape: {target_np.shape}")
print(f"full statistic dimension: {target_flat_t.numel()}")
print(f"active statistic dimension: {mu_np.size}")
print(f"covariance: {COVARIANCE_NPZ}")

target: /Users/tsouros/Desktop/Projects/Nifty8/lss_cov/lss.npy
target shape: (1, 256, 256)
full statistic dimension: 196
active statistic dimension: 188
covariance: /Users/tsouros/Desktop/Projects/Nifty8/lss_cov/lss_covariance.npz


In [5]:
def scattering_vector_with_grad(x_2d: torch.Tensor, st_op, args: Namespace) -> torch.Tensor:
    """Differentiable Phi(x) using the target-derived STL normalization."""
    if x_2d.ndim != 2:
        raise ValueError(f"expected a 2D map, got shape {tuple(x_2d.shape)}")
    data = DataClass(x_2d[None, None, :, :], pbc=args.pbc)
    stats = apply_standardized_st(
        st_op,
        data,
        mean_field=False,
        mark_standardized=False,
        norm="load_ref",
        compute_PS=args.compute_ps,
    )
    flat = stats.to_flatten(
        keep_batch_dim=True,
        mean_along_batch=False,
        keepnans=False,
        flatten_complex=True,
    ).real[0]
    return flat


class _ScatteringWhitenedResidual(ift.Operator):
    """Whitened STL statistic residual used for NIFTy likelihood bookkeeping."""

    def __init__(self, domain, st_op, args, mu_t, precision_sqrt_t, active_mask_t, device, dtype):
        self._domain = ift.makeDomain(domain)
        self._target = ift.UnstructuredDomain(tuple(mu_t.shape))
        self._st_op = st_op
        self._args = args
        self._mu_t = mu_t
        self._precision_sqrt_t = precision_sqrt_t
        self._active_mask_t = active_mask_t
        self._device = device
        self._dtype = dtype

    @staticmethod
    def _as_numeric_array(x_value) -> np.ndarray:
        if hasattr(x_value, "val"):
            x_value = x_value.val
        if isinstance(x_value, dict):
            if len(x_value) != 1:
                raise TypeError(f"expected one field value, got keys {list(x_value)}")
            x_value = next(iter(x_value.values()))
        return np.asarray(x_value)

    def apply(self, x):
        self._check_input(x)
        x_arr = self._as_numeric_array(x.val)
        with torch.no_grad():
            x_t = torch.as_tensor(np.array(x_arr, copy=True), device=self._device, dtype=self._dtype)
            phi = scattering_vector_with_grad(x_t, self._st_op, self._args)[self._active_mask_t]
            residual = self._precision_sqrt_t @ (phi - self._mu_t)
        return ift.makeField(self._target, residual.detach().cpu().numpy())


class ScatteringGaussianEnergy(ift.LikelihoodEnergyOperator):
    """NIFTy scalar energy for 0.5 (Phi(x)-mu)^T C^{-1} (Phi(x)-mu)."""

    def __init__(
        self,
        domain,
        st_op,
        args: Namespace,
        mu: np.ndarray,
        precision: np.ndarray,
        active_mask: np.ndarray,
        *,
        device: torch.device,
        dtype: torch.dtype,
        metric_strength: float = 1.0,
    ):
        self._domain = ift.makeDomain(domain)
        self._st_op = st_op
        self._args = args
        self._device = device
        self._dtype = dtype
        self._active_mask_t = torch.as_tensor(active_mask, device=device, dtype=torch.bool)
        self._mu_t = torch.as_tensor(mu, device=device, dtype=dtype)
        self._precision_t = torch.as_tensor(precision, device=device, dtype=dtype)
        evals, evecs = np.linalg.eigh(0.5 * (precision + precision.T))
        evals = np.clip(evals, 0.0, None)
        precision_sqrt = (evecs * np.sqrt(evals)[None, :]) @ evecs.T
        self._precision_sqrt_t = torch.as_tensor(precision_sqrt, device=device, dtype=dtype)
        self._data_domain = ift.UnstructuredDomain(mu.shape)
        self._res = _ScatteringWhitenedResidual(
            self._domain,
            st_op,
            args,
            self._mu_t,
            self._precision_sqrt_t,
            self._active_mask_t,
            device,
            dtype,
        )
        self._sqrt_data_metric_at = lambda x: ift.ScalingOperator(self._data_domain, 1.0)
        self._name = "scattering"
        self._metric_strength = float(metric_strength)

    @property
    def data_domain(self):
        return self._data_domain

    def normalized_residual(self, x):
        x_arr = self._as_numeric_array(x.val if hasattr(x, "val") else x)
        with torch.no_grad():
            x_t = torch.as_tensor(np.array(x_arr, copy=True), device=self._device, dtype=self._dtype)
            phi = scattering_vector_with_grad(x_t, self._st_op, self._args)[self._active_mask_t]
            residual = self._precision_sqrt_t @ (phi - self._mu_t)
        return ift.makeField(self._data_domain, residual.detach().cpu().numpy())

    def get_transformation(self):
        # This custom STL likelihood is not represented by a NIFTy-native
        # differentiable residual operator. The exact energy/gradient is supplied
        # in apply(); this method exists only for NIFTy likelihood bookkeeping.
        return (np.float64, self._res)

    @staticmethod
    def _as_numeric_array(x_value) -> np.ndarray:
        # NIFTy passes a raw ndarray for plain Field input, but a Field object
        # as Linearization.val during EnergyAdapter/NewtonCG evaluations.
        if hasattr(x_value, "val"):
            x_value = x_value.val
        if isinstance(x_value, dict):
            if len(x_value) != 1:
                raise TypeError(f"expected one field value, got keys {list(x_value)}")
            x_value = next(iter(x_value.values()))
        arr = np.asarray(x_value)
        if arr.dtype == object:
            raise TypeError(f"could not unwrap NIFTy value into a numeric array; got dtype={arr.dtype}")
        return arr

    def _value_and_gradient(self, x_np: np.ndarray):
        x_arr = self._as_numeric_array(x_np)
        x_t = torch.as_tensor(np.array(x_arr, copy=True), device=self._device, dtype=self._dtype)
        x_t.requires_grad_(True)
        phi = scattering_vector_with_grad(x_t, self._st_op, self._args)
        phi = phi[self._active_mask_t]
        diff = phi - self._mu_t
        energy = 0.5 * diff @ (self._precision_t @ diff)
        energy.backward()
        grad = x_t.grad.detach().cpu().numpy().astype(np.float64, copy=False)
        return float(energy.detach().cpu()), grad

    def apply(self, x):
        self._check_input(x)
        value, grad_np = self._value_and_gradient(x.val)
        value_field = ift.Field.scalar(value)
        if x.jac is None:
            return value_field
        grad_field = ift.makeField(self._domain, grad_np)
        jac = VdotOperator(grad_field)
        res = x.new(value_field, jac)
        if x.want_metric:
            metric = ift.ScalingOperator(self._domain, self._metric_strength, sampling_dtype=np.float64)
            res = res.add_metric(metric)
        return res

In [6]:
position_space = ift.RGSpace(tuple(target_np.shape[-2:]))
scattering_energy_x = ScatteringGaussianEnergy(
    position_space,
    target_op,
    args,
    mu_np,
    precision_np,
    active_mask_np,
    device=device,
    dtype=dtype,
    metric_strength=METRIC_STRENGTH,
)

if USE_CORRELATED_FIELD_PRIOR:
    # Optional structured prior: latent standard normal -> correlated field x.
    harmonic_partner = position_space.get_default_codomain()
    cfm = ift.CorrelatedFieldMaker("cf")
    cfm.set_amplitude_total_offset(0.0, (1.0, 0.5))
    cfm.add_fluctuations(
        position_space,
        fluctuations=(1.0, 0.5),
        flexibility=(1.0, 0.5),
        asperity=None,
        loglogavgslope=(-2.0, 0.5),
        harmonic_partner=harmonic_partner,
    )
    signal_op = cfm.finalize(prior_info=0)
else:
    # Direct standard-normal field prior over x. FieldAdapter gives the latent
    # variable a named MultiDomain key, which is required by ift.optimize_kl.
    # The signal itself is still the 2D field on position_space.
    signal_op = ift.FieldAdapter(position_space, "x")

likelihood_energy = scattering_energy_x @ signal_op
print("likelihood domain:", likelihood_energy.domain)
print("signal target:", signal_op.target)

likelihood domain: MultiDomain:
  x: DomainTuple, len: 1
  * RGSpace(shape=(256, 256), distances=(np.float64(0.00390625), np.float64(0.00390625)), harmonic=False)
signal target: DomainTuple, len: 1
* RGSpace(shape=(256, 256), distances=(np.float64(0.00390625), np.float64(0.00390625)), harmonic=False)


In [7]:
def evaluate_map_energy(x_np: np.ndarray) -> float:
    field = ift.makeField(position_space, x_np)
    return float(scattering_energy_x(field).val)

x0_np = np.asarray(target_np[0], dtype=np.float64)
print("E(target[0]) =", evaluate_map_energy(x0_np))
print("target mean/std =", float(x0_np.mean()), float(x0_np.std()))

E(target[0]) = 1316.9296629676535
target mean/std = -6.886835990371765e-08 1.0000000104217983


In [8]:
map_position = None
map_signal = None

if RUN_MAP:
    H = ift.StandardHamiltonian(likelihood_energy)
    # Do not start from the identically-zero map: STL standardization has zero
    # variance there, which makes all scattering entries degenerate.
    start = ift.from_random(likelihood_energy.domain, "normal", std=0.1)
    energy = ift.EnergyAdapter(start, H, want_metric=True, nanisinf=True)
    controller = ift.GradientNormController(tol_abs_gradnorm=1e-5, iteration_limit=30)
    minimizer = ift.NewtonCG(controller, enable_logging=True)
    solution_energy, status = minimizer(energy)
    map_position = solution_energy.position
    map_signal = signal_op(map_position)
    print("MAP status:", status)
    print("MAP Hamiltonian:", float(solution_energy.value))
    print("MAP scattering energy:", float(likelihood_energy(map_position).val))

Iteration limit reached. Assuming convergence


MAP status: 0
MAP Hamiltonian: 1936.9682162506492
MAP scattering energy: 158.36459353586858


In [9]:
posterior_samples = None
posterior_mean = None
posterior_std = None
mean_latent = None

if RUN_VI:
    kl_controller = ift.GradientNormController(tol_abs_gradnorm=1e-5, iteration_limit=25)
    kl_minimizer = ift.NewtonCG(kl_controller, enable_logging=True)
    sampling_ic = ift.GradInfNormController(1e-3, iteration_limit=100)

    if USE_GEOVI:
        nl_controller = ift.GradientNormController(tol_abs_gradnorm=1e-3, iteration_limit=8)
        nonlinear_sampling_minimizer = ift.NewtonCG(nl_controller, enable_logging=False)
    else:
        nonlinear_sampling_minimizer = None

    # NIFTy calls `extra.minisanity` unconditionally after each KL iteration.
    # That diagnostic assumes a fully NIFTy-native likelihood residual; this
    # notebook uses a custom Torch/STL likelihood whose value and gradient are
    # valid, but whose residual is not reliably composable by that diagnostic.
    # Keep the sampler path intact and silence only the diagnostic failure.
    import nifty8.extra as ift_extra

    _original_minisanity = ift_extra.minisanity

    def _stl_safe_minisanity(likelihood_energy, samples, terminal_colors=True, return_values=False):
        try:
            return _original_minisanity(
                likelihood_energy,
                samples,
                terminal_colors=terminal_colors,
                return_values=return_values,
            )
        except (TypeError, ValueError):
            return ("", {}) if return_values else ""

    ift_extra.minisanity = _stl_safe_minisanity
    try:
        posterior_samples, mean_latent = ift.optimize_kl(
            likelihood_energy=likelihood_energy,
            total_iterations=VI_ITERATIONS,
            n_samples=N_POSTERIOR_SAMPLES,
            kl_minimizer=kl_minimizer,
            sampling_iteration_controller=sampling_ic,
            nonlinear_sampling_minimizer=nonlinear_sampling_minimizer,
            plot_energy_history=False,
            plot_minisanity_history=False,
            return_final_position=True,
            sanity_checks=False,
        )
    finally:
        ift_extra.minisanity = _original_minisanity

    posterior_mean, posterior_var = posterior_samples.sample_stat(op=signal_op)
    posterior_std = ift.makeField(
        posterior_var.domain,
        np.sqrt(np.maximum(posterior_var.val, 0.0)),
    )
    print("VI complete")
    print("posterior mean scattering energy:", evaluate_map_energy(posterior_mean.val))

Iteration limit reached. Assuming convergence

Task 0
* apply: 		      0
* apply Linearization: 	   1361
* Jacobian: 		    816
* Adjoint Jacobian: 	   1384
Iteration limit reached. Assuming convergence

Task 0
* apply: 		      0
* apply Linearization: 	   1409
* Jacobian: 		    816
* Adjoint Jacobian: 	   1432


KeyboardInterrupt: 

In [ ]:
# Materialize a small set of posterior signal samples for plotting/saving.
signal_samples = []
if posterior_samples is not None:
    for i, xi in enumerate(posterior_samples.iterator()):
        if i >= N_POSTERIOR_SAMPLES:
            break
        signal_samples.append(signal_op(xi).val)
    signal_samples = np.stack(signal_samples, axis=0) if signal_samples else np.empty((0,) + position_space.shape)
else:
    signal_samples = np.empty((0,) + position_space.shape)

out_path = SCATTERING_VI / "nifty8_scattering_posterior_samples.npz"
np.savez_compressed(
    out_path,
    target_map=target_np,
    posterior_samples=signal_samples,
    posterior_mean=None if posterior_mean is None else posterior_mean.val,
    posterior_std=None if posterior_std is None else posterior_std.val,
    map_signal=None if map_signal is None else map_signal.val,
    mu=mu_np,
    covariance=covariance_np,
    active_statistic_mask=active_mask_np,
)
print("saved", out_path)

In [ ]:
def show_image(ax, image, title, *, vmin=None, vmax=None):
    im = ax.imshow(image, origin="lower", cmap="RdBu_r", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_axis_off()
    return im

reference = np.asarray(target_np[0])
images = [(reference, "target")]
if map_signal is not None:
    images.append((map_signal.val, "MAP"))
if posterior_mean is not None:
    images.append((posterior_mean.val, "VI mean"))
if posterior_std is not None:
    images.append((posterior_std.val, "VI std"))
for i in range(min(4, signal_samples.shape[0])):
    images.append((signal_samples[i], f"sample {i}"))

vmin = float(min(np.percentile(img, 1) for img, _ in images if "std" not in _))
vmax = float(max(np.percentile(img, 99) for img, _ in images if "std" not in _))

ncols = min(4, len(images))
nrows = int(np.ceil(len(images) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax, (image, title) in zip(axes, images):
    if "std" in title:
        show_image(ax, image, title)
    else:
        show_image(ax, image, title, vmin=vmin, vmax=vmax)
for ax in axes[len(images):]:
    ax.set_axis_off()
plt.show()